# CONDA toxicity detection — k=1, **Same-speaker context**

This notebook fine-tunes **DeBERTa-v3-base** at k = 1 (the best context size from the main ablation, macro-F1 = 0.840) but **restricts the context to one type of speaker**.

**Same-speaker context (k = 1)**: the single previous message in the conversation that was sent by the **same player** as the target message. If the target player has no prior message in the conversation, the context is empty.

Tests the hypothesis that *toxicity is consistent within a player over time* — i.e., a player's prior utterance is informative about their current one.

**Comparison group** (already trained, do not rerun): the *all-speaker* k = 1 model in `conda_k1.ipynb` (macro-F1 = 0.840, f1_I = 0.762).

**Output:** saved model directory + meta.json with eval metrics.  
**Runtime:** ~20 min on an A100 GPU.

## 1. Setup

In [ ]:
# Install deps (Colab usually has torch + transformers)
!pip install -q sentencepiece protobuf

In [ ]:
from pathlib import Path

K = 10                              # best k from the main ablation
KIND = 'same_speaker'         # 'same_speaker' or 'other_speaker'

# Local (Colab disk). Change to a Drive path if you want persistence.
SAVE_ROOT = Path('/content/conda_speaker_ablation')
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = SAVE_ROOT / f'k{K}_{KIND}_model'
CKPT_DIR  = SAVE_ROOT / f'k{K}_{KIND}_checkpoints'
MODEL_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)
print(f'Will save model to: {MODEL_DIR}')

Will save model to: /content/conda_speaker_ablation/k10_same_speaker_model


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import json, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert device.type == 'cuda', 'GPU required'

Device: cuda


## 2. Load CONDA dataset

Upload `CONDA_train.csv` and `CONDA_valid.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()

train_df = pd.read_csv('CONDA_train.csv')
valid_df = pd.read_csv('CONDA_valid.csv')
print(f'Train: {train_df.shape}, Valid: {valid_df.shape}')
train_df.head()

Saving CONDA_test.csv to CONDA_test.csv
Saving CONDA_train.csv to CONDA_train.csv
Saving CONDA_valid.csv to CONDA_valid.csv
Train: (26921, 10), Valid: (8974, 10)


,Id,matchId,conversationId,utterance,chatTime,playerSlot,playerId,intentClass,slotClasses,slotTokens
0,11263,697,3193,wow!,76,0,ANTS IN MY EYES JOHNSON,O,O,"wow (O),"
1,13741,843,3809,WTF,1563,5,M.k,O,T,"WTF (T),"
2,22125,1412,6199,wpe wpe,2853,1,Acqua Ragia,O,O O,"wpe (O), wpe (O),"
3,6453,439,1875,hahaha,1038,0,juicebox,O,O,"hahaha (O),"
4,9644,601,2713,wtf,1661,5,KAIST.Shadows,O,T,"wtf (T),"


## 3. Preprocessing

In [ ]:
LABEL2ID = {'E': 0, 'I': 1, 'A': 2, 'O': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

train_df['label'] = train_df['intentClass'].map(LABEL2ID)
valid_df['label'] = valid_df['intentClass'].map(LABEL2ID)

for df in (train_df, valid_df):
    df.sort_values(['conversationId', 'chatTime'], inplace=True, kind='mergesort')
    df.reset_index(drop=True, inplace=True)

print('Label distribution (train):')
print(train_df['label'].value_counts().sort_index().rename(index=ID2LABEL))

Label distribution (train):
label
E     3528
I     1692
A     1719
O    19982
Name: count, dtype: int64


In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

## 4. Speaker-filtered context (k = 1, **Same-speaker**)

For each target message, scan backward through the conversation and take **the most recent message whose player == target player**.

We also report how many target messages actually have any context under this filter.

In [ ]:
def build_speaker_filtered_context(df, kind, max_k=10):
    """For each row, return indices of the up to max_k most-recent prior messages
    in the same conversation, filtered by speaker relationship to the target.

    kind='same_speaker': only messages from the same playerSlot as the target.
    kind='other_speaker': only messages from a different playerSlot than the target.
    """
    assert kind in {'same_speaker', 'other_speaker'}
    same = (kind == 'same_speaker')

    playerSlot = df['playerSlot'].values
    ctx = [[] for _ in range(len(df))]

    for _, group in df.groupby('conversationId', sort=False):
        idxs = group.index.to_numpy()
        for pos, i in enumerate(idxs):
            target_p = playerSlot[i]
            prior = idxs[:pos]
            if same:
                keep = prior[playerSlot[prior] == target_p]
            else:
                keep = prior[playerSlot[prior] != target_p]
            ctx[i] = keep[-max_k:].tolist()
    return ctx

MAX_K = 10
train_context_idx = build_speaker_filtered_context(train_df, KIND, max_k=MAX_K)
valid_context_idx = build_speaker_filtered_context(valid_df, KIND, max_k=MAX_K)

# Diagnostic: how many target messages have any context under this filter?
n_with_ctx = sum(1 for c in train_context_idx if c)
print(f'Train rows with at least 1 {KIND} prior msg: '
      f'{n_with_ctx}/{len(train_df)} ({100*n_with_ctx/len(train_df):.1f}%)')
n_with_ctx_v = sum(1 for c in valid_context_idx if c)
print(f'Valid rows with at least 1 {KIND} prior msg: '
      f'{n_with_ctx_v}/{len(valid_df)} ({100*n_with_ctx_v/len(valid_df):.1f}%)')

Train rows with at least 1 same_speaker prior msg: 8171/26921 (30.4%)
Valid rows with at least 1 same_speaker prior msg: 1343/8974 (15.0%)


In [ ]:
SEP = ' [SEP] '

class CONDASpeakerDataset(Dataset):
    """Concatenate the (filtered) k previous messages with the target.
    Speaker tags are kept so the model can still distinguish authors."""
    def __init__(self, df, context_idx, tokenizer, k, max_length=128):
        self.df = df
        self.context_idx = context_idx
        self.tokenizer = tokenizer
        self.k = k
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def _fmt(self, row, include_speaker):
        text = str(row['utterance'])
        return f"P{int(row['playerSlot'])}: {text}" if include_speaker else text

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ctx_rows = self.context_idx[idx][-self.k:]
        if ctx_rows:
            parts = [self._fmt(self.df.iloc[j], True) for j in ctx_rows]
            parts.append(self._fmt(row, True))
            input_str = SEP.join(parts)
        else:
            # No context available under the filter => fall back to target-only
            input_str = self._fmt(row, True)
        enc = self.tokenizer(
            input_str, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(int(row['label']), dtype=torch.long),
        }

MAX_LENGTH = 128
train_dataset = CONDASpeakerDataset(train_df, train_context_idx, tokenizer, K, MAX_LENGTH)
valid_dataset = CONDASpeakerDataset(valid_df, valid_context_idx, tokenizer, K, MAX_LENGTH)

# Show one sample WITH context and one WITHOUT (to verify both code paths)
with_ctx_i = next((i for i, c in enumerate(train_context_idx) if c), 0)
no_ctx_i   = next((i for i, c in enumerate(train_context_idx) if not c), 0)
for label_str, i in [('WITH context', with_ctx_i), ('WITHOUT context', no_ctx_i)]:
    s = train_dataset[i]
    decoded = tokenizer.decode(s['input_ids'], skip_special_tokens=False)
    print(f'--- Sample {label_str} (label={ID2LABEL[s["labels"].item()]}) ---')
    print(decoded[:300])
    print()

--- Sample WITH context (label=O) ---
P1: wtf [SEPA] TA? [SEPA] u srsly?[SEP] P1: why alyway hit me [SEPA] what did i do[PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PA

--- Sample WITHOUT context (label=I) ---
P6: ez 500[PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD]



## 5. Load model + metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=torch.float32,
).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded ({n_params:,} parameters)')

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

Model loaded (184,425,220 parameters)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per = f1_score(labels, preds, average=None, zero_division=0, labels=[0,1,2,3])
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_E': f1_per[0],
        'f1_I': f1_per[1],
        'f1_A': f1_per[2],
        'f1_O': f1_per[3],
    }

## 6. Train

Same hyperparameters as `conda_k1.ipynb`.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

ckpts = [p for p in CKPT_DIR.glob('checkpoint-*') if p.is_dir()]
resume = bool(ckpts)
if resume:
    print(f'Resuming from {[p.name for p in ckpts]}')

t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=resume)
elapsed_min = (time.time() - t0) / 60
print(f'\nTraining done in {elapsed_min:.1f} min')
print(f'Final train loss: {train_result.training_loss:.4f}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 E,F1 I,F1 A,F1 O
1,0.514474,0.451507,0.881212,0.750245,0.785384,0.658254,0.625000,0.932343
2,0.405311,0.410286,0.907288,0.810400,0.826786,0.721451,0.747273,0.946090
3,0.318853,0.388777,0.912191,0.824584,0.838384,0.726115,0.785080,0.948756
4,0.274361,0.384877,0.916202,0.828835,0.850918,0.728435,0.784452,0.951537
5,0.216969,0.388158,0.917094,0.830599,0.850367,0.734310,0.785396,0.952325


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


Training done in 19.4 min
Final train loss: 0.4170


## 7. Final evaluation

In [ ]:
eval_metrics = trainer.evaluate()
print('Validation metrics (best model):')
for kk, v in eval_metrics.items():
    if isinstance(v, float):
        print(f'  {kk:30s} {v:.4f}')

Validation metrics (best model):
  eval_loss                      0.3882
  eval_accuracy                  0.9171
  eval_f1_macro                  0.8306
  eval_f1_E                      0.8504
  eval_f1_I                      0.7343
  eval_f1_A                      0.7854
  eval_f1_O                      0.9523
  eval_runtime                   22.0288
  eval_samples_per_second        407.3760
  eval_steps_per_second          12.7560
  epoch                          5.0000


In [ ]:
preds_output = trainer.predict(valid_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
print(classification_report(
    y_true, y_pred,
    target_names=['E', 'I', 'A', 'O'],
    digits=4, zero_division=0,
))

              precision    recall  f1-score   support

           E     0.8680    0.8335    0.8504      1183
           I     0.9385    0.6031    0.7343       582
           A     0.8122    0.7603    0.7854       580
           O     0.9322    0.9733    0.9523      6629

    accuracy                         0.9171      8974
   macro avg     0.8877    0.7926    0.8306      8974
weighted avg     0.9164    0.9171    0.9140      8974



## 8. Compare against the all-speaker baseline at k = 1

Reference (from `conda_k1.ipynb`, all-speaker context):

| class | precision | recall | f1 |
|---|---|---|---|
| E | 0.8265 | 0.8740 | 0.8496 |
| I | 0.8770 | 0.6735 | 0.7619 |
| A | 0.8071 | 0.7862 | 0.7965 |
| O | 0.9464 | 0.9581 | 0.9522 |
| **macro** | **0.8642** | **0.8230** | **0.8401** |

If this run beats the all-speaker baseline on macro-F1, the speaker-filter carries more signal than just having any context. If it underperforms, the immediate predecessor (regardless of who said it) was already capturing the useful signal.

## 9. Save model + metadata

In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

meta = {
    'k': K,
    'context_kind': KIND,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'train_minutes': elapsed_min,
    'final_train_loss': float(train_result.training_loss),
    'eval_metrics': {kk: float(v) for kk, v in eval_metrics.items() if isinstance(v, (int, float))},
    'train_rows_with_context': sum(1 for c in train_context_idx if c),
    'valid_rows_with_context': sum(1 for c in valid_context_idx if c),
    'train_total_rows': len(train_df),
    'valid_total_rows': len(valid_df),
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'sep_token': SEP.strip(),
    'speaker_tag_format': 'P{playerSlot}:',
}
with open(MODEL_DIR / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved to {MODEL_DIR}')
for p in sorted(MODEL_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:30s} {size_mb:8.2f} MB')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/conda_speaker_ablation/k10_same_speaker_model
  config.json                        0.00 MB
  meta.json                          0.00 MB
  model.safetensors                737.73 MB
  tokenizer.json                     8.34 MB
  tokenizer_config.json              0.00 MB
  training_args.bin                  0.01 MB


In [ ]:
# Optional: clean intermediate checkpoints
import shutil
for ckpt in CKPT_DIR.glob('checkpoint-*'):
    if ckpt.is_dir():
        shutil.rmtree(ckpt)
        print(f'Removed {ckpt.name}')

Removed checkpoint-6732
Removed checkpoint-8415


## 10. Download archive (Colab)

In [ ]:
import tarfile, os
from google.colab import files

archive_path = f'/content/k{K}_{KIND}_model.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(str(MODEL_DIR), arcname=f'k{K}_{KIND}_model')

print(f'Archive: {archive_path} ({os.path.getsize(archive_path)/1e6:.1f} MB)')
files.download(archive_path)

Archive: /content/k10_same_speaker_model.tar.gz (586.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>